In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [5]:
# Download SMS Spam dataset
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"

df = pd.read_csv(url, sep="\t", header=None, names=["label", "message"])

df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
print("Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nLabel Distribution:")
print(df["label"].value_counts())

print("\nMissing Values:")
print(df.isnull().sum())

Dataset Shape: (5572, 2)

Columns:
Index(['label', 'message'], dtype='object')

Label Distribution:
label
ham     4825
spam     747
Name: count, dtype: int64

Missing Values:
label      0
message    0
dtype: int64


In [7]:
# Convert labels into numerical values
df["label_num"] = df["label"].map({"ham": 0, "spam": 1})

df.head()

,label,message,label_num
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [8]:
# X = input messages
# y = target labels

X = df["message"]
y = df["label_num"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (5572,)
y shape: (5572,)


In [9]:
# Split data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 4457
Testing samples: 1115


In [10]:
# Convert text messages into numerical features using TF-IDF

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english"
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Training data shape:", X_train_tfidf.shape)
print("Testing data shape:", X_test_tfidf.shape)

Training data shape: (4457, 7403)
Testing data shape: (1115, 7403)


In [11]:
# Train Logistic Regression classifier

model = LogisticRegression(max_iter=1000)

model.fit(X_train_tfidf, y_train)

print("Model training completed successfully!")

Model training completed successfully!


In [12]:
# Predict spam/ham labels for test messages

y_pred = model.predict(X_test_tfidf)

print("Predictions completed!")
print("First 10 predictions:", y_pred[:10])

Predictions completed!
First 10 predictions: [0 0 0 1 0 0 0 0 0 0]


In [13]:
# Calculate model accuracy

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Accuracy Percentage:", round(accuracy * 100, 2), "%")

Accuracy: 0.967713004484305
Accuracy Percentage: 96.77 %


In [14]:
# Generate classification report

print(classification_report(
    y_test,
    y_pred,
    target_names=["Ham", "Spam"]
))

              precision    recall  f1-score   support

         Ham       0.96      1.00      0.98       966
        Spam       1.00      0.76      0.86       149

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.97      1115



In [15]:
# Test the model with new messages

new_messages = [
    "Congratulations! You won a free lottery ticket. Call now!",
    "Hey, are we meeting tomorrow for lunch?",
    "URGENT! You have won a cash prize. Claim it now!"
]

new_messages_tfidf = vectorizer.transform(new_messages)

predictions = model.predict(new_messages_tfidf)

for message, prediction in zip(new_messages, predictions):
    result = "SPAM" if prediction == 1 else "HAM"
    print(f"Message: {message}")
    print(f"Prediction: {result}")
    print("-" * 60)

Message: Congratulations! You won a free lottery ticket. Call now!
Prediction: HAM
------------------------------------------------------------
Message: Hey, are we meeting tomorrow for lunch?
Prediction: HAM
------------------------------------------------------------
Message: URGENT! You have won a cash prize. Claim it now!
Prediction: SPAM
------------------------------------------------------------


In [16]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[966   0]
 [ 36 113]]


In [17]:
# Create a summary of the NLP classification project

results = {
    "Model": "Logistic Regression",
    "Technique": "TF-IDF",
    "Accuracy": round(accuracy * 100, 2)
}

results_df = pd.DataFrame([results])

print(results_df)

                 Model Technique  Accuracy
0  Logistic Regression    TF-IDF     96.77


In [18]:
# Save results as CSV

results_df.to_csv("NLP_Text_Classification_Results.csv", index=False)

print("Results file created successfully!")

Results file created successfully!
